# 모델 양자화 (Quantization) - 실습 코드 1: AWQ 양자화 모델 로드 및 추론

- Tutorial ID: `expand-quantization`
- Tutorial: 모델 양자화 (Quantization)
- Section ID: `expand-quantization-code-1`
- Section: 실습 코드 1: AWQ 양자화 모델 로드 및 추론


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: AWQ 양자화 모델 로드 및 추론
#
# 이 코드는 "실행 버튼만 누르고 끝내는" 용도가 아니라,
# "양자화를 하면 메모리가 줄어든다"는 말이 실제로는 어떤 숫자로 확인되는지
# 한 줄씩 직접 측정하면서 따라가기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) AWQ 4비트 양자화 모델을 실제로 불러오고, GPU 메모리를 직접 측정해서
#      FP16 원본 대비 얼마나 줄어드는지 눈으로 확인한다.
#   2) 4비트로 압축된 모델도 정상적으로 문장을 이해하고 생성할 수 있음을 확인한다.
#   3) 토크나이저 → 모델 로드 → 토큰화 → 생성 → 디코딩으로 이어지는
#      추론(inference) 파이프라인 전체 흐름을 이해한다.
#
# 읽는 순서:
#   1) 실습 환경(GPU 여부, 필요 라이브러리)을 먼저 확인합니다.
#   2) 모델을 불러오기 "전"과 "후"의 GPU 메모리를 각각 측정해서 비교합니다.
#   3) device_map, dtype 같은 주요 인자가 각각 무엇을 하는지 확인합니다.
#   4) 프롬프트가 토큰화 → 생성 → 디코딩되는 과정을 단계별 출력으로 확인합니다.
#   5) 프롬프트, max_new_tokens, temperature 등을 바꿔가며 결과가 어떻게
#      달라지는지 직접 실험해봅니다. (맨 아래 "6. 더 해보기" 참고)
#
# 주의:
#   - 이 노트북은 GPU(CUDA) 환경에서 실행하는 것을 전제로 합니다.
#     Colab이라면 [런타임] > [런타임 유형 변경]에서 GPU를 선택하세요.
#   - autoawq 등 양자화 관련 라이브러리는 업데이트가 빠르고, 그중 일부는
#     유지보수가 중단된 상태이기도 합니다. 설치나 실행 중 오류가 발생하면
#     라이브러리 버전 문제일 가능성이 높으니, 본문의 버전 관련 안내를
#     참고하세요.
#   - 코드에 나오는 숫자(메모리 용량, 속도 등)를 외우려 하지 말고,
#     "이론적으로 계산한 값"과 "직접 측정한 값"이 왜, 얼마나 다른지에
#     집중해서 보세요.
# ============================================================


## 0. 시작하기 전에: 양자화(Quantization)와 AWQ가 뭔가요?

**양자화(Quantization)**는 모델이 내부에 가지고 있는 수많은 숫자(가중치, weight)를 더 적은 비트(bit) 수로 표현해서, 모델이 차지하는 저장 공간과 메모리 사용량을 줄이는 기법입니다.

### 비유로 이해하기: 캐리어에 짐 싸기

해외여행을 갈 때, 정해진 캐리어 용량(= GPU 메모리)에 짐(= 모델의 파라미터)을 모두 넣어야 한다고 생각해봅시다.

- **원본 모델(FP16)**은 옷을 압축백 없이 그대로 넣는 것과 비슷합니다. 형태는 그대로 보존되지만 캐리어 공간을 많이 차지합니다.
- **양자화**는 옷을 압축백에 눌러 담는 것과 비슷합니다. 부피는 훨씬 줄어들지만, 너무 세게 누르면(너무 적은 비트를 쓰면) 옷의 원래 모양(모델의 정확도)이 망가질 수 있습니다.

### AWQ는 무엇이 다른가요?

모든 숫자를 똑같은 세기로 압축하면, 결과에 큰 영향을 주는 "중요한" 숫자까지 함께 뭉개질 수 있습니다. 캐리어에 짐을 쌀 때도 노트북이나 안경처럼 깨지기 쉬운 물건은 조심히 감싸고, 양말이나 티셔츠는 아무렇게나 눌러 담아도 되는 것처럼요.

**AWQ(Activation-aware Weight Quantization)**는 바로 이 "어떤 숫자가 중요한지"를, 모델이 실제로 동작할 때 내부에 흘러가는 값(activation, 활성화 값)을 관찰해서 판단합니다. 중요하다고 판단된 가중치는 정밀도를 최대한 보존하고, 덜 중요한 가중치는 과감하게 압축해서, 전체 크기는 크게 줄이면서도 성능 저하는 최소화하는 방법입니다. ("Activation-aware"라는 이름 자체가 "활성화 값을 고려한다"는 뜻입니다.)

### 비트(bit) 수와 용량의 관계

컴퓨터가 숫자 하나를 저장할 때 몇 비트를 쓰는지에 따라 필요한 용량이 달라집니다.

| 정밀도 | 숫자 하나당 크기 | 파라미터 70억(7B) 개 모델의 예상 크기 |
|---|---|---|
| FP32 (32비트) | 4바이트 | 약 28GB |
| FP16 (16비트) | 2바이트 | 약 14GB |
| INT8 (8비트) | 1바이트 | 약 7GB |
| **INT4 (4비트, AWQ)** | **0.5바이트** | **약 3.5GB** |

이번 실습에서 사용할 모델은 Meta의 **Llama-2-7B-Chat**(파라미터 약 70억 개)을 AWQ 방식으로 4비트 양자화한 버전입니다. 이론적으로는 FP16 대비 약 1/4 크기(약 3.5GB)까지 줄어들어야 합니다. 아래에서 직접 불러와서 실제로 얼마나 줄어드는지 측정해보겠습니다.


## 1. 실습 환경 준비

이 실습은 다음 환경을 전제로 합니다.

- **GPU(CUDA) 환경**: AWQ 양자화 모델은 GPU 전용 연산 커널(kernel)을 사용하기 때문에, GPU가 없는 환경(CPU만 있는 환경)에서는 정상적으로 동작하지 않거나 매우 느립니다. Google Colab을 사용한다면 상단 메뉴에서 **[런타임] > [런타임 유형 변경]**에서 하드웨어 가속기를 **GPU**로 설정해주세요.
- **필요 라이브러리**
  - `transformers` : 모델과 토크나이저를 불러오는 핵심 라이브러리
  - `accelerate` : 모델을 GPU 등 여러 장치에 자동으로 배치해주는 라이브러리
  - `autoawq` : AWQ 4비트 양자화 모델을 실제로 GPU에서 계산(추론)할 수 있게 해주는 라이브러리

> 💡 **참고 1**: `autoawq` 라이브러리는 2025년 5월부터 더 이상 새 버전이 개발되지 않는(deprecated) 상태입니다. 다만 이미 배포된 버전은 계속 정상적으로 동작하고, 이번 실습처럼 "이미 양자화된 모델을 불러와서 추론"하는 용도에는 문제가 없습니다. 다만 `autoawq`를 설치하면 그 과정에서 `transformers` 버전이 예전 버전(4.47.1)으로 자동으로 낮아질 수 있습니다. 혹시 이후 다른 코드에서 최신 `transformers` 기능이 필요하다면, `autoawq` 설치 후 `pip install -U transformers`로 다시 최신 버전으로 올려주세요.
>
> 💡 **참고 2**: 아래에서 사용할 GPU 메모리 측정 코드는 **GPU가 1개인 환경**을 기준으로 합니다. 여러 GPU를 함께 쓰는 환경이라면 측정 결과가 다르게 보일 수 있습니다.


In [ ]:
# 처음 한 번만 실행하면 됩니다. (이미 설치되어 있다면 다시 실행해도 문제없습니다)
#   - transformers : 허깅페이스(Hugging Face)의 모델·토크나이저를 불러오는 핵심 라이브러리
#   - accelerate   : device_map="auto"처럼 모델을 GPU 등에 자동으로 배치해주는 라이브러리
#   - autoawq      : AWQ 4비트 양자화 모델을 실제로 GPU에서 계산(추론)하기 위한 라이브러리
!pip install -q transformers accelerate autoawq

# 설치 로그에서 transformers 버전이 낮아졌다는 메시지가 보였다면,
# 아래 줄의 주석(#)을 지우고 실행해서 다시 최신 버전으로 맞출 수 있습니다.
# !pip install -q -U transformers


In [ ]:
import time     # 텍스트 생성 속도(초당 토큰 수)를 재기 위한 파이썬 표준 라이브러리
import torch    # 텐서 연산과 GPU 메모리 확인에 사용하는 PyTorch 라이브러리

from transformers import AutoTokenizer, AutoModelForCausalLM
# - AutoTokenizer        : 텍스트를 모델이 이해하는 숫자(토큰 id)로 바꿔주는 도구를 자동으로 찾아 불러옵니다.
# - AutoModelForCausalLM : "지금까지 나온 단어들을 보고 다음 단어를 예측"하는 방식의 언어모델을 자동으로 불러옵니다.
#                          (Causal LM = 앞에서부터 순서대로 다음 토큰을 예측하는 언어모델)

print("라이브러리 불러오기 완료")


In [ ]:
# AWQ 양자화 모델은 GPU 전용 연산 커널을 사용하기 때문에 GPU(CUDA)가 반드시 필요합니다.
# 아래 코드는 지금 이 노트북이 GPU를 인식하고 있는지 미리 확인합니다.
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"사용 가능한 GPU: {gpu_name}")
    print(f"GPU 전체 메모리: {total_memory_gb:.1f} GB")
else:
    print("GPU(CUDA)를 찾을 수 없습니다.")
    print("Colab이라면 [런타임] > [런타임 유형 변경] > 하드웨어 가속기를 'GPU'로 바꾼 뒤,")
    print("다시 처음(라이브러리 설치)부터 실행해주세요.")


## 2. AWQ 양자화 모델 불러오기

이 실습에서는 `TheBloke/Llama-2-7B-Chat-AWQ` 모델을 사용합니다.

- **Llama-2-7B-Chat**: Meta에서 공개한 파라미터 약 70억(7B) 개 규모의 대화형(Chat) 언어모델입니다.
- **TheBloke**: 다양한 오픈소스 언어모델을 양자화해서 공개해온 커뮤니티 기여자입니다. 원본 Llama-2-7B-Chat 모델을 미리 AWQ 4비트로 양자화해서 올려두었기 때문에, 우리가 직접 양자화 작업을 하지 않고도 바로 불러와서 사용할 수 있습니다. (모델을 처음부터 양자화하는 과정 자체는 이번 실습 범위가 아닙니다.)

모델을 불러올 때 사용하는 주요 인자(argument)는 다음과 같습니다.

| 인자 | 의미 |
|---|---|
| `device_map="auto"` | 모델의 각 레이어를 사용 가능한 장치(GPU/CPU)에 자동으로 배치합니다. GPU가 1개면 그 GPU에, 여러 개면 자동으로 나누어 올립니다. |
| `dtype=torch.float16` | 양자화되지 않은 나머지 연산(레이어 정규화 등)에 사용할 부동소수점 정밀도를 지정합니다. (예전 코드에서는 `torch_dtype`이라는 이름을 쓰기도 했는데, 최신 transformers 버전에서는 `dtype`이라는 이름으로 바뀌었습니다. `torch_dtype`을 써도 동작은 하지만 "곧 없어질 예정"이라는 경고 메시지가 뜹니다.) |

> 💡 **참고**: 혹시 모델을 내려받는(다운로드) 과정에서 인증 관련 오류(401 등)가 발생한다면, 터미널이나 셀에서 `huggingface-cli login`을 실행해 허깅페이스 계정으로 로그인한 뒤 다시 시도해보세요.


In [ ]:
model_name = "TheBloke/Llama-2-7B-Chat-AWQ"

# 토크나이저는 크기가 작아서(수 MB 수준) GPU 메모리에 거의 영향을 주지 않습니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("토크나이저 로드 완료")

# --- 본격적으로 모델을 불러오기 전에, GPU 메모리 사용량을 "기준점(0에 가까운 값)"으로 만들어 둡니다. ---
# empty_cache()             : 더 이상 쓰지 않는 캐시 메모리를 비웁니다.
# reset_peak_memory_stats() : 지금까지 기록된 "최대 사용량" 기록을 초기화합니다.
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

memory_before_gb = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"모델 로드 전 GPU 메모리 사용량: {memory_before_gb:.2f} GB")


In [ ]:
# 실제로 AWQ 4비트 양자화 모델을 GPU에 불러옵니다.
# 모델 크기에 따라 다운로드 + 로드에 1~수 분 정도 걸릴 수 있습니다.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",     # 모델을 사용 가능한 GPU에 자동으로 배치
    dtype=torch.float16,   # 양자화되지 않은 연산은 FP16 정밀도로 수행
)

print("AWQ 양자화 모델 로드 완료")
print(f"모델이 올라간 위치: {model.device}")


In [ ]:
# 모델을 다 올린 "직후"의 GPU 메모리 사용량을 측정합니다.
memory_after_gb = torch.cuda.memory_allocated() / (1024 ** 3)

# "로드 후" 값에서 "로드 전" 값을 빼면, 이 모델이 순수하게 차지한 메모리를 알 수 있습니다.
model_memory_gb = memory_after_gb - memory_before_gb

print(f"모델 로드 후 GPU 메모리 사용량: {memory_after_gb:.2f} GB")
print(f"이 AWQ 모델이 실제로 차지한 메모리: 약 {model_memory_gb:.2f} GB")
print()

# --- 이론적으로 계산한 값과 비교해봅니다 ---
num_params_billion = 7  # Llama-2-7B는 파라미터가 약 70억 개입니다.

fp16_theory_gb = num_params_billion * 2      # FP16: 파라미터 1개당 2바이트
int4_theory_gb = num_params_billion * 0.5    # INT4: 파라미터 1개당 0.5바이트(4비트)

print("[이론적으로 계산한 크기]")
print(f"  - FP16(원본) 예상 크기 : 약 {fp16_theory_gb:.1f} GB")
print(f"  - INT4(AWQ) 예상 크기  : 약 {int4_theory_gb:.1f} GB")
print()
print("[방금 실제로 측정한 크기]")
print(f"  - AWQ 모델 로드 후 실제 GPU 메모리 사용량 : 약 {model_memory_gb:.2f} GB")
print()
print("실제 측정치는 이론치와 정확히 일치하지 않고 조금 더 큰 경우가 많습니다.")
print("그룹 단위(예: 128개씩 묶어서)로 원래 값을 복원할 때 필요한")
print("scale(축척 값)·zero-point(기준점 값)를 별도로 저장해두는 오버헤드,")
print("그리고 CUDA 구동에 필요한 기본 컨텍스트 메모리 등이 더해지기 때문입니다.")


## 3. 프롬프트를 토큰으로 바꾸기 (토큰화)

언어모델은 문자를 그대로 이해하지 못하고, 미리 정해진 "토큰(token)" 단위의 숫자(id)로 바꿔서 입력받습니다. `tokenizer`가 바로 이 변환(텍스트 → 숫자)을 담당합니다.

예를 들어 `"Explain"`이라는 단어 하나가 통째로 토큰 1개가 될 수도 있고, 모델이 학습한 방식에 따라 여러 조각(sub-word)으로 쪼개질 수도 있습니다. 아래에서 실제로 어떻게 쪼개지는지 직접 눈으로 확인해봅니다.


In [ ]:
prompt = "Explain quantum computing in simple terms:"

# return_tensors="pt" : 결과를 PyTorch 텐서(tensor) 형태로 반환합니다.
# .to(model.device)   : 입력 텐서를 모델이 올라가 있는 장치(GPU)로 옮깁니다.
#                        "cuda"라고 직접 쓰는 대신 model.device를 쓰면,
#                        device_map으로 모델이 여러 GPU에 나뉘어 올라간 경우에도 안전합니다.
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print(f"원본 프롬프트: {prompt}")
print()
print(f"토큰 id: {inputs['input_ids']}")
print(f"토큰 개수: {inputs['input_ids'].shape[1]}개")
print()

# 토큰 id가 실제로 어떤 문자/조각에 대응하는지 하나씩 확인해봅니다.
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"토큰 분해 결과: {tokens}")


## 4. 텍스트 생성하기

이제 토큰화된 입력을 모델에 넣어, 다음에 올 토큰들을 순서대로 예측(생성)하도록 시켜봅니다. `generate()` 함수에서 자주 쓰는 인자는 다음과 같습니다.

| 인자 | 의미 |
|---|---|
| `max_new_tokens` | 새로 생성할 토큰의 최대 개수입니다. 입력 프롬프트 길이와는 별개로, "새로 만들어낼" 토큰 수만 제한합니다. |
| `do_sample` | `False`면 매 단계마다 확률이 가장 높은 토큰 하나만 선택하는 결정적(greedy) 방식이라 같은 입력에는 항상 같은 결과가 나옵니다. `True`면 확률 분포를 따라 무작위로 선택하는 샘플링 방식이라 실행할 때마다 결과가 조금씩 달라집니다. |
| `pad_token_id` | 여러 문장을 배치로 묶어 길이를 맞출 때 쓰는 padding 토큰의 id입니다. Llama-2 토크나이저는 기본적으로 pad 토큰이 따로 없어서, 문장 종료를 뜻하는 eos 토큰으로 대신 지정해줍니다. |

먼저 `do_sample=False`(결정적 방식)로 실행해서 재현 가능한 결과를 확인해보고, 이후 "6. 더 해보기"에서 `do_sample=True`도 함께 실험해봅니다.


In [ ]:
# 생성을 시작하기 전, "생성 과정에서만" 추가로 사용되는 메모리를 따로 측정하기 위해
# peak(최대 사용량) 기록을 다시 초기화합니다.
torch.cuda.reset_peak_memory_stats()

start_time = time.time()

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,                       # 매 단계 확률이 가장 높은 토큰을 선택 (재현 가능한 결과)
    pad_token_id=tokenizer.eos_token_id,    # Llama-2는 기본 pad 토큰이 없어 eos 토큰으로 대체
)

elapsed_sec = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

num_input_tokens = inputs["input_ids"].shape[1]
num_new_tokens = outputs.shape[1] - num_input_tokens

print(f"생성에 걸린 시간: {elapsed_sec:.2f}초")
print(f"새로 생성된 토큰 수: {num_new_tokens}개")
print(f"초당 생성 토큰 수: {num_new_tokens / elapsed_sec:.2f} tokens/sec")
print(f"생성 중 추가로 사용된 최대 GPU 메모리: 약 {peak_memory_gb:.2f} GB")
print("(모델 가중치 자체는 이미 로드되어 있으므로, 이 수치에는")
print(" 생성 과정에서 새로 필요해진 KV 캐시·중간 연산 메모리가 포함됩니다.)")


In [ ]:
# outputs[0]에는 "입력 프롬프트 토큰 + 새로 생성된 토큰"이 그대로 이어져 있습니다.
# 그래서 그대로 디코딩하면 프롬프트 내용이 앞부분에 그대로 다시 나타납니다.
print("=" * 60)
print("[전체 출력: 프롬프트 + 생성된 텍스트]")
print("=" * 60)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
# skip_special_tokens=True : <s>, </s> 처럼 사람이 읽을 필요 없는 특수 토큰을 결과에서 숨깁니다.

print()
print("=" * 60)
print("[새로 생성된 부분만 보기]")
print("=" * 60)
# 입력 토큰 길이만큼 잘라내면, 모델이 "새로" 만들어낸 부분만 남습니다.
generated_ids = outputs[0][num_input_tokens:]
print(tokenizer.decode(generated_ids, skip_special_tokens=True))


In [ ]:
print("=" * 60)
print("실습 결과 요약")
print("=" * 60)
print("[메모리]")
print(f"  - FP16 이론적 예상 크기               : 약 {fp16_theory_gb:.1f} GB")
print(f"  - AWQ(4비트) 이론적 예상 크기          : 약 {int4_theory_gb:.1f} GB")
print(f"  - AWQ 모델 실제 측정 크기(가중치)      : 약 {model_memory_gb:.2f} GB")
print(f"  - 텍스트 생성 중 최대(peak) 메모리 사용량 : 약 {peak_memory_gb:.2f} GB")
print()
print("[속도]")
print(f"  - 총 {num_new_tokens}개 토큰을 {elapsed_sec:.2f}초 만에 생성")
print(f"  - 초당 약 {num_new_tokens / elapsed_sec:.2f} 토큰 생성")
print("=" * 60)


## 5. 정리: 무엇을 확인했나요?

이번 실습에서 직접 실행하고 측정한 내용을 정리하면 다음과 같습니다.

- **메모리**: Llama-2-7B를 FP16으로 올리면 이론상 약 14GB가 필요하지만, AWQ 4비트로 양자화된 버전은 훨씬 적은 메모리로 로드됩니다. 정확한 수치는 바로 위 "실습 결과 요약" 셀의 실행 결과를 확인하세요. (실행할 때마다, 그리고 GPU 종류에 따라 값이 조금씩 달라질 수 있습니다.)
- **정상 동작 확인**: 4비트로 압축된 모델도 정상적으로 문장을 이해하고 자연스러운 텍스트를 생성할 수 있음을 확인했습니다.
- **속도**: 초당 몇 개의 토큰을 생성하는지도 함께 측정했습니다. 같은 모델이라도 GPU 종류, 프롬프트 길이, `max_new_tokens` 값에 따라 속도는 달라질 수 있습니다.

> ⚠️ **주의**: 이번 실습은 "이미 양자화되어 배포된 모델을 불러와서 추론하는 과정"입니다. 원본 FP16 모델을 직접 AWQ 4비트로 변환(양자화)하는 과정 자체는 다루지 않았습니다.


## 6. 더 해보기 (직접 실험해보기)

아래처럼 값을 바꿔가며 다시 실행해보면, 양자화와 추론 과정을 더 깊이 이해할 수 있습니다.

1. **프롬프트 바꾸기**: 위 "3. 프롬프트를 토큰으로 바꾸기" 셀의 `prompt` 변수를 다른 질문으로 바꾸고, 이어지는 셀들을 다시 실행해보세요.
2. **생성 길이 바꾸기**: `max_new_tokens`을 50, 500 등으로 바꿔보고, 생성 시간과 "생성 중 최대 GPU 메모리"가 어떻게 달라지는지 관찰해보세요.
3. **샘플링 켜보기**: `do_sample=False`를 `do_sample=True, temperature=0.7`로 바꿔서 같은 프롬프트를 여러 번 실행해보세요. `do_sample=False`일 때와 달리, 실행할 때마다 결과가 조금씩 달라지는 것을 확인할 수 있습니다.
4. **(참고) 채팅 형식 프롬프트**: 이번 실습에서는 프롬프트를 그대로 모델에 넣었지만, Llama-2-Chat 계열 모델은 `[INST] 질문 내용 [/INST]` 형태의 지시문(instruction) 형식으로 감싸주면 더 자연스러운 대화형 답변을 얻는 경우가 많습니다. 예: `f"[INST] {prompt} [/INST]"`
5. **(선택, 고급) FP16 원본과 비교**: GPU 메모리가 24GB 이상으로 넉넉하다면, `del model`과 `torch.cuda.empty_cache()`로 지금 모델을 메모리에서 내린 뒤, 양자화되지 않은 원본 모델(`meta-llama/Llama-2-7b-chat-hf`, 접근 권한 신청 필요)을 같은 방식으로 불러와서 이번에 측정한 AWQ 모델의 메모리·속도와 직접 비교해볼 수 있습니다.
